<a href="https://colab.research.google.com/github/sergiocostaifes/PPCOMP_DM/blob/main/notebooks/04_window_5min_episodes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04_window_5min_episodes.ipynb — Detecção de Episódios Críticos (5 min)

# Objetivo
Detectar automaticamente **episódios críticos** na série temporal agregada em janelas fixas de **5 minutos**, utilizando o limiar estatístico **μ + 2σ** aplicado sobre a **taxa de falha** (**fail_rate**), consolidando intervalos contínuos e gerando métricas por episódio.

## Entradas (artefatos do Notebook 03)
- `window_5min_base.parquet`
- `window_5min_series.parquet` (**preferencial**: série reindexada e contínua)

## Saídas (artefatos deste Notebook)
- `episodes_detected.parquet`
- `04_detect_episodes_summary.json`

## Compatibilidade com o pipeline existente (00 → 03)
Este notebook é compatível com a estrutura gerada no Notebook 03, que define:
- eixo temporal: `bucket_id` (inteiro) e `bucket_start_us` (microssegundos)
- métricas por janela: contagens agregadas (incluindo `n_failed` e `n_events`)
- série contínua: `window_5min_series.parquet` com preenchimento de buckets ausentes com 0 nas métricas de contagem

## Definições
- Janela temporal: **5 min** (300 s)
- Métrica oficial para detecção: **`fail_rate = n_failed / n_events`**
  - Tratamento de borda: quando `n_events = 0`, define-se `fail_rate = 0.0`
- Limiar estatístico: **`threshold = μ + 2σ`** (calculado sobre `fail_rate`)
- Janela crítica: **`fail_rate >= threshold`**
- Episódio: sequência contínua de janelas críticas (adjacentes em `bucket_id`)

## Métricas por episódio
**Estruturais**
- `episode_id`
- `start_bucket`, `end_bucket`
- `duration_windows`, `duration_minutes` (= `duration_windows * 5`)

**Intensidade (oficial — taxa)**
- `max_fail_rate`, `mean_fail_rate`

**Diagnóstico (volume absoluto)**
- `max_failed`, `mean_failed`, `sum_failed`

**Parâmetros do limiar (da taxa)**
- `threshold`, `mu`, `sigma`

## Observações de reprodutibilidade
- Bootstrap via Drive + repositório `PPCOMP_DM`
- Uso de `src.paths` e `ensure_dirs()`
- Execução independente de caminhos relativos frágeis
- Logs resumidos no final e persistência em `03-features` e `04-reports`

In [1]:
# ============================================================
# 04_window_5min_episodes.ipynb
# Detecção de episódios críticos (μ + 2σ) na série 5-min
# Métrica oficial: taxa de falha (fail_rate = n_failed / n_events)
# Pipeline PPCOMP_DM (Google Cluster Trace)
# ============================================================

# -----------------------------
# 0) BOOTSTRAP (Colab + Repo)
# -----------------------------
from pathlib import Path
import os
import sys
import subprocess
import importlib
import random
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Mount Google Drive (seguro)
if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("[Bootstrap] Google Drive já montado.")

# Ajuste aqui se o seu repo estiver em outro caminho do Drive
REPO_DIR = Path("/content/drive/MyDrive/Mestrado/PPCOMP_DM")
GITHUB_REPO = "https://github.com/sergiocostaifes/PPCOMP_DM.git"

if not REPO_DIR.exists():
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    print(f"[Bootstrap] Clonando repositório em: {REPO_DIR}")
    subprocess.run(["git", "clone", GITHUB_REPO, str(REPO_DIR)], check=True)
else:
    # Atualiza repo sem quebrar notebook se houver alterações locais
    try:
        print("[Bootstrap] Atualizando repositório (git pull)...")
        subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
    except Exception as e:
        print("[Bootstrap] Aviso: não foi possível atualizar via git pull:", e)

# Trabalhar sempre a partir do repo (evita path relativo quebrar)
os.chdir(str(REPO_DIR))
print("[Bootstrap] CWD =", os.getcwd())

# Garantir import do pacote src/
repo_str = str(REPO_DIR)
if repo_str not in sys.path:
    sys.path.insert(0, repo_str)
importlib.invalidate_caches()

# Paths padronizados
from src.paths import PROCESSED_PATH, FEATURES_PATH, REPORTS_PATH, ensure_dirs
ensure_dirs()

print("PROCESSED_PATH =", PROCESSED_PATH)
print("FEATURES_PATH =", FEATURES_PATH)
print("REPORTS_PATH  =", REPORTS_PATH)

def log(msg: str) -> None:
    print(f"[04_detect_episodes] {msg}")

# =========================
# 1) Leitura da série contínua
# =========================


import json

SERIES_FILE = FEATURES_PATH / "window_5min_series.parquet"
assert SERIES_FILE.exists(), f"Arquivo não encontrado: {SERIES_FILE}"

df = pd.read_parquet(SERIES_FILE)

log(f"Shape entrada: {df.shape}")

# colunas mínimas
assert "bucket_id" in df.columns
assert "n_failed" in df.columns
assert "n_events" in df.columns  # novo: denominador da taxa

df = df.sort_values("bucket_id").reset_index(drop=True)

# =========================
# 2) Métrica oficial: taxa de falha (fail_rate)
# =========================
# tratamento de borda: n_events = 0 => fail_rate = 0.0
den = df["n_events"].replace(0, np.nan)
df["fail_rate"] = (df["n_failed"] / den).fillna(0.0).astype("float32")

# =========================
# 3) Estatística global (na taxa)
# =========================
mu = df["fail_rate"].mean()
sigma = df["fail_rate"].std(ddof=0)
threshold = mu + 2 * sigma

log(f"mu={mu:.8f}")
log(f"sigma={sigma:.8f}")
log(f"threshold={threshold:.8f}")

df["is_critical"] = (df["fail_rate"] >= threshold).astype("int8")
critical_windows = int(df["is_critical"].sum())
log(f"Total janelas críticas: {critical_windows}")

# =========================
# 4) Consolidação de episódios (contiguidade em bucket_id)
# =========================
episodes = []
in_episode = False
start_idx = None

for i, flag in enumerate(df["is_critical"]):
    if flag == 1 and not in_episode:
        in_episode = True
        start_idx = i
    elif flag == 0 and in_episode:
        episodes.append((start_idx, i - 1))
        in_episode = False

if in_episode:
    episodes.append((start_idx, len(df) - 1))

log(f"Episódios detectados: {len(episodes)}")

# =========================
# 5) Métricas por episódio
# =========================
WINDOW_MINUTES = 5

rows = []

for ep_id, (s, e) in enumerate(episodes, start=1):
    seg = df.iloc[s:e+1]

    rows.append({
        "episode_id": ep_id,
        "start_bucket": int(seg["bucket_id"].iloc[0]),
        "end_bucket": int(seg["bucket_id"].iloc[-1]),
        "duration_windows": int(len(seg)),
        "duration_minutes": int(len(seg) * WINDOW_MINUTES),

        # oficial (taxa)
        "max_fail_rate": float(seg["fail_rate"].max()),
        "mean_fail_rate": float(seg["fail_rate"].mean()),

        # diagnóstico (volume absoluto)
        "max_failed": int(seg["n_failed"].max()),
        "mean_failed": float(seg["n_failed"].mean()),
        "sum_failed": int(seg["n_failed"].sum()),

        # parâmetros do limiar (da taxa)
        "threshold": float(threshold),
        "mu": float(mu),
        "sigma": float(sigma),
    })

episodes_df = pd.DataFrame(rows)

log(f"Episodes DF shape: {episodes_df.shape}")

# =========================
# 6) Persistência
# =========================
OUT_FILE = FEATURES_PATH / "episodes_detected.parquet"
episodes_df.to_parquet(OUT_FILE, compression="snappy", index=False)

summary = {
    "metric": "fail_rate",
    "series_rows": int(len(df)),
    "episodes_detected": int(len(episodes_df)),
    "critical_windows": int(critical_windows),
    "mu": float(mu),
    "sigma": float(sigma),
    "threshold": float(threshold),
}

summary_file = REPORTS_PATH / "04_detect_episodes_summary.json"
summary_file.write_text(json.dumps(summary, indent=2, ensure_ascii=False))

log("Notebook 04 finalizado com sucesso.")
episodes_df.head()

Mounted at /content/drive
[Bootstrap] Atualizando repositório (git pull)...
[Bootstrap] CWD = /content/drive/MyDrive/Mestrado/PPCOMP_DM
PROCESSED_PATH = /content/drive/MyDrive/Mestrado/02-datasets/02-processed
FEATURES_PATH = /content/drive/MyDrive/Mestrado/02-datasets/03-features
REPORTS_PATH  = /content/drive/MyDrive/Mestrado/04-reports
[04_detect_episodes] Shape entrada: (8918, 18)
[04_detect_episodes] mu=0.19650584
[04_detect_episodes] sigma=0.14087607
[04_detect_episodes] threshold=0.47825798
[04_detect_episodes] Total janelas críticas: 394
[04_detect_episodes] Episódios detectados: 285
[04_detect_episodes] Episodes DF shape: (285, 13)
[04_detect_episodes] Notebook 04 finalizado com sucesso.


,episode_id,start_bucket,end_bucket,duration_windows,duration_minutes,max_fail_rate,mean_fail_rate,max_failed,mean_failed,sum_failed,threshold,mu,sigma
0,1,666,666,1,5,0.500000,0.500000,7,7.0,7,0.478258,0.196506,0.140876
1,2,967,967,1,5,0.756757,0.756757,84,84.0,84,0.478258,0.196506,0.140876
2,3,1220,1220,1,5,0.625000,0.625000,10,10.0,10,0.478258,0.196506,0.140876
3,4,1391,1391,1,5,0.750000,0.750000,6,6.0,6,0.478258,0.196506,0.140876
4,5,1567,1567,1,5,0.600000,0.600000,39,39.0,39,0.478258,0.196506,0.140876


## Achados do Notebook 04 — Resultados e Sanidade (taxa)

### Entrada e consistência temporal
- Artefato utilizado: `window_5min_series.parquet`
- Total de janelas na série contínua: **8918**
- A série está ordenada por `bucket_id` e é adequada para consolidação de intervalos contíguos.

### Métrica principal (taxa) e regra de criticidade
- Métrica adotada para detecção: **`fail_rate = n_failed / n_events`**
- Tratamento de borda: quando `n_events = 0`, define-se `fail_rate = 0.0` (evita divisão por zero).
- A detecção foi feita com base em `fail_rate`, usando o limiar estatístico `μ + 2σ`:
  - **μ = 0.1965058446**
  - **σ = 0.1408760697**
  - **threshold = 0.4782579839**

### Janelas críticas e episódios
- Janelas críticas (`fail_rate ≥ threshold`): **394**
- Episódios críticos (intervalos contíguos de janelas críticas): **285**
- Observação: a diferença entre 394 janelas críticas e 285 episódios indica predominância de episódios curtos (frequentemente 1 janela), o que é consistente com picos localizados em uma série agregada por janela fixa.

### Estrutura do dataset de saída
O arquivo `episodes_detected.parquet` foi gerado com **285** linhas (um registro por episódio) e contém:

**Campos estruturais**
- limites do episódio (`start_bucket`, `end_bucket`)
- duração (`duration_windows`, `duration_minutes`)

**Intensidade (oficial — taxa)**
- `max_fail_rate`, `mean_fail_rate`

**Diagnóstico (volume absoluto)**
- `max_failed`, `mean_failed`, `sum_failed`

**Parâmetros do limiar (da taxa)**
- `threshold`, `mu`, `sigma`

### Persistência e rastreabilidade
- Dataset de episódios salvo em: `03-features/episodes_detected.parquet`
- Sumário salvo em: `04-reports/04_detect_episodes_summary.json`
- Métrica oficial registrada no sumário: **`metric = fail_rate`**